# 03 - Vùng cắt Nguyên bản (Native Crops) và Dấu vết Nội suy (Resampling)

Trong bài học này, chúng ta khảo sát ảnh hưởng của phép resize:
- So sánh trực quan trên tập TRAIN Fold 0 giữa 4 patch **Native (1:1)** và 4 patch **Resampled ($112 \rightarrow 224$)**.
- Cơ chế **Multi-crop Mean Logits Pooling**: Lấy trung bình điểm logit qua 4 crop trước khi đưa qua hàm sigmoid.
- Huấn luyện đối chứng trên Fold 0: `native` vs `resampled` (control).
- Phân tích điều kiện kiểm soát và giới hạn phương pháp.

In [ ]:
from pathlib import Path
import os
import sys

def find_task_root():
    cwd = Path.cwd().resolve()
    for cand in [cwd, cwd.parent, cwd / 'KeMaoDanh', cwd.parent / 'KeMaoDanh']:
        if (cand / 'src/kmd').is_dir() and (cand / 'configs').is_dir():
            return cand.resolve()
    raise FileNotFoundError("Mở notebook từ repo root, KeMaoDanh hoặc KeMaoDanh/notebooks.")

TASK_ROOT = find_task_root()
if str(TASK_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(TASK_ROOT / 'src'))

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from torchvision.transforms import functional as TF

from kmd.core import PACKAGE, read_csv, split_fold, metric, read_json
from kmd.config import Config
from kmd.pipeline import prepare_development, load_session, train_cnn_fold

RUN_ID = 'lesson_session'
print(f"Phiên làm việc: {RUN_ID}")

## 1. Trực quan hóa 4 vị trí crop: Native vs Resampled

Ta lấy một ảnh từ tập **TRAIN Fold 0** để quan sát 4 vị trí crop cố định:
- Trung tâm $(0.5, 0.5)$, Góc trên trái $(0.38, 0.38)$, Góc trên phải $(0.62, 0.38)$, Góc dưới $(0.5, 0.64)$.

“Native 1:1” nghĩa là một pixel crop lấy từ một pixel ảnh nguồn, không resize trước khi vào mạng. Control giữ đúng vùng đó rồi giảm 224→112 và phóng lại 224. Việc này có thể loại chi tiết hữu ích hoặc loại nhiễu; ta chưa biết hướng tác động. Hình dưới chỉ minh họa can thiệp trên train, còn kết luận so sánh phải dựa trên validation.

In [ ]:
data_root_env = os.environ.get('DATA_ROOT')
data_root = Path(data_root_env or TASK_ROOT / 'data/train').expanduser().resolve()
if not (data_root / 'pairs.csv').is_file():
    raise FileNotFoundError(f'Thiếu dữ liệu train: {data_root / "pairs.csv"}. Xem README để đặt DATA_ROOT.')

if (data_root / 'pairs.csv').is_file():
    dev_frame = prepare_development(data_root)
    tr_fold0, va_fold0 = split_fold(dev_frame, fold=0)
    sample_path = data_root / tr_fold0.iloc[0].image_0
    
    if sample_path.is_file():
        with Image.open(sample_path) as im:
            im = im.convert('RGB')
            w, h = im.size
            centers = [(0.5, 0.5), (0.38, 0.38), (0.62, 0.38), (0.5, 0.64)]
            native_patches, resampled_patches = [], []
            
            for cx, cy in centers:
                left = max(0, min(w - 224, round(cx * w) - 112))
                top = max(0, min(h - 224, round(cy * h) - 112))
                patch = im.crop((left, top, left + 224, top + 224))
                native_patches.append(patch)
                
                # Resample: 224 -> 112 -> 224
                res_patch = TF.resize(TF.resize(patch, [112, 112], antialias=True), [224, 224], antialias=True)
                resampled_patches.append(res_patch)
                
        fig, axes = plt.subplots(2, 4, figsize=(15, 7))
        for i in range(4):
            axes[0, i].imshow(native_patches[i], interpolation='nearest')
            axes[0, i].set_title(f"Crop {i+1}: Native 224x224")
            axes[0, i].axis('off')
            
            axes[1, i].imshow(resampled_patches[i], interpolation='nearest')
            axes[1, i].set_title(f"Crop {i+1}: Resampled (112->224)")
            axes[1, i].axis('off')
        fig.suptitle("Hàng trên = Native 1:1 (Gốc) | Hàng dưới = Resampled (Nội suy)", fontsize=13)
        plt.tight_layout()
        plt.show()

## 2. Cơ chế Multi-crop Mean Logits Pooling

Với mỗi ảnh có 4 crops:
1. Mô hình xuất ra 4 logits: $s^{(1)}, s^{(2)}, s^{(3)}, s^{(4)}$.
2. Logit đại diện của ảnh là trung bình cộng: $s = \frac{1}{4} \sum_{i=1}^4 s^{(i)}$.
3. Xác suất cặp ảnh: $p = \sigma(s_1 - s_0)$.

Lưu ý rằng $\sigma\left(\frac{1}{4}\sum s^{(i)}\right) \neq \frac{1}{4}\sum \sigma(s^{(i)})$ do hàm sigmoid phi tuyến tính. Việc lấy trung bình ở không gian logits bảo toàn thang đo tuyến tính trước khi nén về $[0, 1]$.

## 3. Huấn luyện Đối chứng Fold 0: Native vs Resampled

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("Bước huấn luyện CNN cần CUDA. Bạn có thể chạy nhánh LR ở bài 01 và 05 trên CPU.")

if (data_root / 'pairs.csv').is_file() and torch.cuda.is_available():
    session_dir = load_session(RUN_ID, data_root)
    print(f"Nạp session tại: {session_dir.name}")
    
    # 1. Huấn luyện Native Fold 0
    folder_native, res_native = train_cnn_fold('native', dev_frame, data_root, session_dir, fold=0)
    
    # 2. Huấn luyện Resampled Fold 0
    folder_resampled, res_resampled = train_cnn_fold('resampled', dev_frame, data_root, session_dir, fold=0)
    
    print("\n=== KẾT QUẢ ĐỐI CHỨNG VALIDATION FOLD 0 ===")
    df_eval = pd.DataFrame([
        {'Cấu hình': '1. Native (4 native patches 224x224)', **res_native['metrics']},
        {'Cấu hình': '2. Resampled Control (112->224)', **res_resampled['metrics']},
    ])
    print(df_eval[['Cấu hình', 'macro_f1', 'accuracy', 'log_loss', 'errors']].to_string(index=False))
else:
    print("Huấn luyện yêu cầu GPU CUDA và dataset tại data/train.")

### Theo dõi điểm của từng crop trên một cặp validation

Sau khi train, lấy ID đầu tiên theo thứ tự cố định, không chọn theo độ đúng. Bảng dưới cho thấy 8 logit đi tới một xác suất cặp như thế nào. Checkpoint vừa được wrapper kiểm tra trước khi nạp; chỉ nạp file do chính phiên này tạo.

In [ ]:
from kmd.models import model_for, logits
from kmd.data import Pairs

state = torch.load(folder_native / 'best.pt', map_location='cpu', weights_only=False)
c = Config(**state['config'])
model, _ = model_for(c, device='cpu', pretrained=False)
model.load_state_dict(state['state_dict'], strict=True)
model.eval()
row = va_fold0.sort_values('pair_id').head(1)
x, y = Pairs(row, data_root, c, norm=state.get('norm'))[0]
x = x.unsqueeze(0)
with torch.inference_mode():
    # [1 cặp, 2 ảnh, 4 crops, 3, 224, 224] -> 8 ảnh -> 2 x 4 logits
    crop_scores = model(x.reshape(-1, *x.shape[-3:])).reshape(2, 4)
    mean_scores = crop_scores.mean(dim=1)
    shared_scores = logits(model, x, c)[0]
    assert torch.allclose(mean_scores, shared_scores, atol=1e-6)
    pair_p = torch.sigmoid(mean_scores[1] - mean_scores[0]).item()
    mean_crop_p = torch.sigmoid(crop_scores[1] - crop_scores[0]).mean().item()
print('ID / nhãn:', row.iloc[0].pair_id, int(y))
print(pd.DataFrame(crop_scores.numpy(), index=['image_0', 'image_1'], columns=['crop1', 'crop2', 'crop3', 'crop4']))
print({'mean_logits': mean_scores.tolist(), 'p_from_mean_logits': pair_p,
       'mean_of_crop_pair_probabilities_for_comparison': mean_crop_p})
del model, state

## 4. Phân tích điều kiện kiểm soát

- **Yếu tố can thiệp:** Thao tác giảm độ phân giải xuống $112 \times 112$ rồi phóng to lại $224 \times 224$.
- **Yếu tố kiểm soát:** Cùng kiến trúc DenseNet-121, cùng 4 tọa độ crop, cùng bộ siêu tham số và số epoch (19 epoch cố định).
- **Hạn chế chung:** Cả hai cấu hình đều chỉ quan sát 4 ô cục bộ, không có cái nhìn toàn cảnh về bố cục bức ảnh.

Trong bài tiếp theo, chúng ta sẽ khảo sát mô hình EfficientNet-B2 với ảnh $288 \times 288$ và phân tích sự kết hợp giữa các mô hình.